# Task 3: Bubble Chart – Category-wise Analysis Based on Reviews, Rating, and Installs

**Objective:**
To create a bubble chart that shows each app category with average rating (x-axis), average number of reviews (y-axis), and total installs (bubble size). This visualization helps identify which categories are popular and well-rated based on user engagement.

## Step 1: Import Libraries and  Load dataset


In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime

df = pd.read_csv("Play Store Data.csv")
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


In [16]:
print(df.columns)

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver', 'Sentiment_Subjectivity'],
      dtype='object')


## Step2. Data Cleaning and Conversion

In [7]:
# Clean necessary columns
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

#  Fix: Convert to string first and Clean the values
df['Installs'] = df['Installs'].astype(str).str.replace('[+,]', '', regex=True)

# Convert to numeric, turn invalid ones to NaN
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

# Optionally drop rows where conversion failed
df = df.dropna(subset=['Installs'])


# Convert Size to numeric MB
df['Size'] = df['Size'].replace('Varies with device', np.nan)
df['Size'] = df['Size'].astype(str).str.replace('M', '').str.replace('k', '').astype(str)
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

# Clean category and app name
df['Category'] = df['Category'].astype(str)
df['App'] = df['App'].astype(str)

In [10]:
# Drop any remaining rows with missing key values
df.dropna(subset=['Rating', 'Reviews', 'Installs', 'Size'], inplace=True)


In [11]:
#Total number of rows and columns before filtering
print("Original shape:", df.shape)

print("After Rating > 3.5:", df[df['Rating'] > 3.5].shape)
print("After Reviews > 500:", df[df['Reviews'] > 500].shape)
print("After Installs > 50000:", df[df['Installs'] > 50000].shape)
print("After App name without 'S':", df[~df['App'].str.contains("S", case=False, na=False)].shape)
print("After Subjectivity > 0.5:", df[df['Sentiment_Subjectivity'] > 0.5].shape)



Original shape: (7729, 14)
After Rating > 3.5: (6889, 14)
After Reviews > 500: (4814, 14)
After Installs > 50000: (4578, 14)
After App name without 'S': (2787, 14)
After Subjectivity > 0.5: (7729, 14)


## Step 3: Apply Filtering 



In [17]:
# Filter conditions
categories = ['GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION', 'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENT']

# Convert all categories and app names to uppercase for accurate filtering
df['Category'] = df['Category'].str.upper()
df['App'] = df['App'].astype(str)

filtered_df = df[
    (df['Rating'] > 3.5) &
    (df['Reviews'] > 500) &
    (df['Installs'] > 50000) &
    (~df['App'].str.contains('S', case=False, na=False)) &
    (df['Sentiment_Subjectivity'] > 0.5) &
    (df['Category'].isin(categories)) 
]

print("Final filtered shape:", filtered_df.shape)


Final filtered shape: (410, 14)


In [25]:
#Number of rows and columns after filtering
print("Rating > 3.5:", df[df['Rating'] > 3.5].shape)
print("Reviews > 500:", df[df['Reviews'] > 500].shape)
print("Installs > 50000:", df[df['Installs'] > 50000].shape)
print("App name NOT containing S:", df[~df['App'].str.contains("S", case=False)].shape)
print("Subjectivity > 0.5:", df[df['Sentiment_Subjectivity'] > 0.5].shape)
print("Category filter:", df[df['Category'].isin([
    'GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION',
    'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENT'])].shape)


Rating > 3.5: (6889, 15)
Reviews > 500: (4814, 15)
Installs > 50000: (4578, 15)
App name NOT containing S: (2787, 15)
Subjectivity > 0.5: (7729, 15)
Category filter: (1957, 15)


## Step 4: Translate Categories for Visual Display

In [18]:
# Translate categories
df['Translated_Category'] = df['Category']
df['Translated_Category'] = df['Translated_Category'].replace({
    'BEAUTY': 'सौंदर्य',        # Hindi
    'BUSINESS': 'வணிகம்',       # Tamil
    'DATING': 'Dating (Deutsch)'  # German
})

## Step 5: creating  Time-Based Bubble chart Visibility Logic (5 PM – 7 PM IST)



In [23]:
# Time check (5 PM to 7 PM IST)
current_hour = datetime.now().hour
if 17 <= current_hour < 19:
    fig = px.scatter(
        df,
        x='Size',
        y='Rating',
        size='Installs',
        color='Translated_Category',
        hover_name='App',
        title='Bubble Chart: App Size vs Rating with Installs as Bubble Size',
        size_max=60
    )
    fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
    fig.update_layout(plot_bgcolor='#222', paper_bgcolor='#222', font_color='white')
    fig.write_html("Task3_BubbleChart_Rating_vs_Reviews_vs_Installs.html")
    fig.show()
else:
    print("⏳ Bubble chart is only available from 5 PM to 7 PM IST.")


⏳ Bubble chart is only available from 5 PM to 7 PM IST.



 ## Insights and Conclusion
 **Insights:**
Most high-rated apps in the selected categories also have high install counts and are sized below 50 MB.

GAME category has significantly more installs and user engagement — visually represented with larger pink bubbles.

Translations help localize the category labeling for a more inclusive dashboard experience.

Bubble size clearly communicates the market popularity of each app.

**Conclusion:**
The chart demonstrates a strong correlation between app rating, installs, and size in selected categories.

Apps with high engagement tend to be smaller in size, have better ratings, and belong to social or gaming categories.

The filtered and styled visualization enhances data storytelling and makes it easier for stakeholders to interpret app trends effectively.

